# RAG Quality Evaluation with Token Factory

**The problem this notebook solves:**  
Nebius Token Factory's cookbook shows you how to *build* a RAG pipeline. But how do you know if it's actually working? Is it hallucinating? Are the answers faithful to the documents? Are the retrieved chunks relevant?

This notebook adds the missing piece — an **automated RAG evaluation pipeline** that scores your RAG system on:
- **Faithfulness** — is the answer grounded in the retrieved context, or is it hallucinating?
- **Answer Relevance** — does the answer actually address the question?
- **Context Precision** — did we retrieve the right chunks?

All evaluation is done using Token Factory's own inference API — no external tools needed.

> **Note:** This notebook was built and tested using Groq's OpenAI-compatible API.  
> To switch to Token Factory, change `base_url` to `https://api.tokenfactory.nebius.com`  
> and update `model` to any Token Factory model (e.g. `meta-llama/Meta-Llama-3.1-70B-Instruct`).

---

## Step 1 — Install dependencies

In [ ]:
!pip install openai faiss-cpu sentence-transformers pandas numpy -q

## Step 2 — Configuration

Set your API key and choose your provider.  
Switch `USE_TOKEN_FACTORY = True` when running on Nebius.

In [ ]:
import os

# ---------- CONFIG ----------
USE_TOKEN_FACTORY = False  # Set True to use Nebius Token Factory

if USE_TOKEN_FACTORY:
    API_KEY   = "YOUR_NEBIUS_API_KEY"
    BASE_URL  = "https://api.tokenfactory.nebius.com"
    MODEL     = "meta-llama/Meta-Llama-3.1-70B-Instruct"
else:
    API_KEY   = "YOUR_GROQ_API_KEY"   # <-- paste your Groq key here
    BASE_URL  = "https://api.groq.com/openai/v1"
    MODEL     = "llama-3.1-8b-instant"
# ----------------------------

print(f"Using: {'Nebius Token Factory' if USE_TOKEN_FACTORY else 'Groq (Token Factory compatible)'}")
print(f"Model: {MODEL}")

## Step 3 — Build a sample knowledge base

We use a small set of documents about AI infrastructure.  
In production, replace this with your own documents.

In [ ]:
# Sample knowledge base — replace with your own documents
DOCUMENTS = [
    "Nebius Token Factory is a managed inference platform that serves 60+ open-source models including Llama, DeepSeek, Qwen, and Mistral. It handles over 100 million tokens per minute with 99.9% uptime SLA.",
    "Token Factory supports fine-tuning workflows. Users can adapt open-source models to their own data and deploy custom checkpoints directly on Token Factory endpoints with per-token pricing.",
    "RAG (Retrieval-Augmented Generation) on Token Factory uses high-performance embedding models and PGVector storage. The platform keeps indexing, retrieval, and inference within one governed environment.",
    "Token Factory pricing is transparent — charged per token with clear input/output separation and volume discounts. There are no hidden GPU or idle costs. Dedicated endpoints include 99.9% SLA and custom autoscaling.",
    "Nebius Token Factory is SOC 2 Type II, HIPAA, and ISO 27001 certified. It operates in zero-retention mode — requests and outputs are never stored or reused for training.",
    "Speculative decoding on Token Factory reduces latency by predicting likely next tokens using a smaller draft model, then verifying with the main model in parallel. This cuts response time without changing output quality.",
    "Token Factory supports multimodal models for text, code, vision, and reasoning tasks. All modalities are served through one unified OpenAI-compatible API endpoint.",
    "Nebius recently acquired Eigen AI to strengthen Token Factory's inference optimization stack. Eigen AI's quantization and batching techniques are now integrated into production model serving.",
    "Enterprise customers on Token Factory get dedicated Slack support, custom DPAs, SSO, RBAC, and unified billing across teams and projects.",
    "Token Factory's Data Lab workspace lets teams explore inference logs, curate datasets, and move directly into model fine-tuning iteration without switching platforms."
]

print(f"Knowledge base loaded: {len(DOCUMENTS)} documents")

## Step 4 — Build FAISS vector index

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load embedding model
# On Token Factory, swap this with their embedding API:
# model = "BAAI/bge-en-icl" via Token Factory embeddings endpoint
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded")

# Embed documents
doc_embeddings = embedder.encode(DOCUMENTS, convert_to_numpy=True)
dimension = doc_embeddings.shape[1]

# Build FAISS index
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

print(f"FAISS index built: {index.ntotal} vectors, dimension {dimension}")

## Step 5 — RAG pipeline

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

def retrieve(query: str, top_k: int = 3) -> list[str]:
    """Retrieve top_k most relevant documents for a query."""
    query_vec = embedder.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_vec, top_k)
    return [DOCUMENTS[i] for i in indices[0]]

def generate(query: str, context_docs: list[str]) -> str:
    """Generate an answer given query and retrieved context."""
    context = "\n\n".join([f"[{i+1}] {doc}" for i, doc in enumerate(context_docs)])
    prompt = f"""Answer the question using ONLY the context below. 
If the answer is not in the context, say 'I don't know based on the provided context.'

Context:
{context}

Question: {query}
Answer:"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=300
    )
    return response.choices[0].message.content.strip()

def rag(query: str) -> dict:
    """Full RAG pipeline: retrieve + generate."""
    context = retrieve(query)
    answer  = generate(query, context)
    return {"query": query, "context": context, "answer": answer}

# Quick test
test = rag("What is Token Factory's uptime SLA?")
print("Query:", test["query"])
print("Answer:", test["answer"])

## Step 6 — Evaluation functions

We use the LLM itself as a judge — a technique called **LLM-as-a-judge**.  
This is the same approach used by RAGAS and other evaluation frameworks,  
but implemented directly on Token Factory with no external dependencies.

In [ ]:
import json

def llm_judge(prompt: str) -> float:
    """Ask the LLM to score something 0.0-1.0. Returns float."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=10
    )
    raw = response.choices[0].message.content.strip()
    try:
        score = float(raw.split()[0].replace(",", "."))
        return min(max(score, 0.0), 1.0)
    except:
        return 0.5  # fallback if LLM doesn't return a clean number


def score_faithfulness(answer: str, context: list[str]) -> float:
    """
    Faithfulness: Is every claim in the answer supported by the context?
    Score 1.0 = fully grounded, 0.0 = hallucinating.
    """
    ctx = "\n".join(context)
    prompt = f"""Rate how faithful the answer is to the context on a scale of 0.0 to 1.0.
1.0 = every claim in the answer is directly supported by the context.
0.0 = the answer contains claims not found in the context (hallucination).

Context: {ctx}
Answer: {answer}

Reply with a single number between 0.0 and 1.0 only."""
    return llm_judge(prompt)


def score_answer_relevance(query: str, answer: str) -> float:
    """
    Answer Relevance: Does the answer actually address the question?
    Score 1.0 = directly answers the question, 0.0 = completely off topic.
    """
    prompt = f"""Rate how relevant the answer is to the question on a scale of 0.0 to 1.0.
1.0 = the answer directly and completely addresses the question.
0.0 = the answer is completely off topic or doesn't address the question.

Question: {query}
Answer: {answer}

Reply with a single number between 0.0 and 1.0 only."""
    return llm_judge(prompt)


def score_context_precision(query: str, context: list[str]) -> float:
    """
    Context Precision: Did we retrieve the right chunks?
    Score 1.0 = all retrieved chunks are relevant, 0.0 = none are relevant.
    """
    scores = []
    for chunk in context:
        prompt = f"""Rate how relevant this context chunk is to answering the question on a scale of 0.0 to 1.0.
1.0 = the chunk directly helps answer the question.
0.0 = the chunk is completely irrelevant to the question.

Question: {query}
Context chunk: {chunk}

Reply with a single number between 0.0 and 1.0 only."""
        scores.append(llm_judge(prompt))
    return sum(scores) / len(scores)


def evaluate(query: str, answer: str, context: list[str]) -> dict:
    """Run all three evaluations and return scores."""
    return {
        "faithfulness":      round(score_faithfulness(answer, context), 2),
        "answer_relevance":  round(score_answer_relevance(query, answer), 2),
        "context_precision": round(score_context_precision(query, context), 2),
    }

print("Evaluation functions ready")

## Step 7 — Run evaluation on test questions

In [ ]:
# Test questions — mix of answerable and unanswerable
TEST_QUESTIONS = [
    "What is Token Factory's uptime SLA?",
    "How does Token Factory handle data privacy and retention?",
    "What fine-tuning options does Token Factory support?",
    "How does speculative decoding reduce latency?",
    "What certifications does Nebius Token Factory have?",
    "Can I deploy my own custom fine-tuned model on Token Factory?",
    "What is the pricing model for Token Factory?",
    "What is the CEO of Nebius's favourite food?",  # unanswerable — tests hallucination
    "Does Token Factory support multimodal models?",
    "What enterprise features does Token Factory offer?",
]

print(f"Running RAG evaluation on {len(TEST_QUESTIONS)} questions...\n")

results = []
for i, question in enumerate(TEST_QUESTIONS):
    print(f"[{i+1}/{len(TEST_QUESTIONS)}] {question[:60]}...")
    rag_result = rag(question)
    scores     = evaluate(question, rag_result["answer"], rag_result["context"])
    results.append({
        "question":          question,
        "answer":            rag_result["answer"],
        "faithfulness":      scores["faithfulness"],
        "answer_relevance":  scores["answer_relevance"],
        "context_precision": scores["context_precision"],
        "avg_score":         round((scores["faithfulness"] + scores["answer_relevance"] + scores["context_precision"]) / 3, 2)
    })

print("\nDone!")

## Step 8 — Results report

In [ ]:
import pandas as pd

df = pd.DataFrame(results)

# Display full results
pd.set_option('display.max_colwidth', 60)
print("=" * 80)
print("RAG EVALUATION REPORT")
print("=" * 80)
print(df[["question", "faithfulness", "answer_relevance", "context_precision", "avg_score"]].to_string(index=False))
print()

In [ ]:
# Summary stats
print("=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"Questions evaluated:    {len(df)}")
print(f"Avg Faithfulness:       {df['faithfulness'].mean():.2f}  (hallucination risk: {'LOW' if df['faithfulness'].mean() > 0.8 else 'HIGH'})")
print(f"Avg Answer Relevance:   {df['answer_relevance'].mean():.2f}")
print(f"Avg Context Precision:  {df['context_precision'].mean():.2f}")
print(f"Overall RAG Score:      {df['avg_score'].mean():.2f} / 1.00")
print()

# Flag weak spots
weak = df[df["avg_score"] < 0.6]
if len(weak) > 0:
    print(f"⚠️  {len(weak)} question(s) scored below 0.6 — investigate these:")
    for _, row in weak.iterrows():
        print(f"   - {row['question'][:70]} (score: {row['avg_score']})")
else:
    print("✅ All questions scored above 0.6")

In [ ]:
# Visualize results
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = 'white'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart — per question scores
x = range(len(df))
width = 0.25
axes[0].bar([i - width for i in x], df["faithfulness"],      width, label="Faithfulness",      color="#4CAF50")
axes[0].bar([i         for i in x], df["answer_relevance"],  width, label="Answer Relevance",  color="#2196F3")
axes[0].bar([i + width for i in x], df["context_precision"], width, label="Context Precision", color="#FF9800")
axes[0].set_xticks(list(x))
axes[0].set_xticklabels([f"Q{i+1}" for i in x])
axes[0].set_ylim(0, 1.1)
axes[0].set_title("RAG Evaluation Scores per Question")
axes[0].set_ylabel("Score (0-1)")
axes[0].axhline(y=0.6, color="red", linestyle="--", alpha=0.5, label="Min threshold (0.6)")
axes[0].legend()

# Radar-style avg scores
categories = ["Faithfulness", "Answer\nRelevance", "Context\nPrecision"]
values     = [df["faithfulness"].mean(), df["answer_relevance"].mean(), df["context_precision"].mean()]
colors     = ["#4CAF50", "#2196F3", "#FF9800"]
bars = axes[1].bar(categories, values, color=colors, width=0.4)
axes[1].set_ylim(0, 1.1)
axes[1].set_title("Average Scores Across All Questions")
axes[1].set_ylabel("Average Score (0-1)")
axes[1].axhline(y=0.6, color="red", linestyle="--", alpha=0.5, label="Min threshold")
for bar, val in zip(bars, values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.02, f"{val:.2f}", ha="center", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.savefig("rag_evaluation_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved as rag_evaluation_results.png")

## Step 9 — How to switch to Token Factory

This notebook is built on Groq's OpenAI-compatible API.  
Switching to Nebius Token Factory requires changing **3 lines only**:

```python
# Change these in Step 2:
USE_TOKEN_FACTORY = True
API_KEY  = "YOUR_NEBIUS_API_KEY"
BASE_URL = "https://api.tokenfactory.nebius.com"
MODEL    = "meta-llama/Meta-Llama-3.1-70B-Instruct"

# For embeddings, swap sentence-transformers with Token Factory's embedding API:
# Model: BAAI/bge-en-icl via Token Factory embeddings endpoint
```

Everything else stays identical — same evaluation logic, same scoring, same report.

---

## What's next

- **Extend the knowledge base** — replace `DOCUMENTS` with your own PDFs or data
- **Add ground truth** — if you have reference answers, add exact match scoring
- **Run on Token Factory's embedding models** — swap `sentence-transformers` with `BAAI/bge-en-icl` via the Token Factory embeddings API for a fully managed pipeline
- **CI integration** — run this notebook as part of your deployment pipeline to catch RAG quality regressions before they reach production